# Level 1 — Classic CNNs (VGG-16, ResNet-18 / 50)

**목표**: VGG 와 ResNet 을 직접 구현하여 Set A 위에서 Multi-task 로 학습하고, **Skip Connection** 이 깊은 네트워크의 수렴에 미치는 영향을 분석합니다.

**금지 사항**: `torchvision.models.*`, `timm`. 단, 위 라이브러리의 코드를 *참고하여 직접 타이핑* 하는 것은 허용됩니다.

본 노트북의 산출물:
1. 학습된 체크포인트 (`checkpoints/level1_*.pth`).
2. VGG-16, ResNet-18, ResNet-50 의 `Avg-Macro-F1` 및 속성별 Macro-F1 표.
3. VGG (skip 없음) vs ResNet (skip 있음) 의 학습 손실 곡선 비교.

In [1]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
#repo_name = "2026-HYU-AUE8088-PA2"
#if not os.path.exists(f"/content/{repo_name}"):
#    !git clone https://github.com/IRCVLab/2026-HYU-AUE8088-PA2.git

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
#%cd /content/{repo_name}

%cd /content
!rm -rf 2026-HYU-AUE8088-PA2
!git clone https://github.com/Jeong-dahun/2026-HYU-AUE8088-PA2.git
%cd /content/2026-HYU-AUE8088-PA2

/content
Cloning into '2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 52 (delta 15), reused 45 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 57.15 KiB | 886.00 KiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/2026-HYU-AUE8088-PA2


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.resnet import resnet18, resnet50
from src.models.vgg import VGG16

SEED = 42
set_seed(SEED, deterministic=True)   # 재현성을 위한 시드 고정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 92.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 117.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 22.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Weights & Biases 설정

학습 곡선과 속성별 Macro-F1 을 자동으로 로깅합니다. 사용하지 않으려면 `WANDB_PROJECT = None` 으로 두세요 — 로깅 호출은 자동으로 no-op 이 됩니다.

최초 1회만:
```python
import wandb; wandb.login()   # API key 입력
```

In [4]:
import wandb; wandb.login()   # API key 입력


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dahun0902 (dahun0902-kookmin-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
WANDB_TAGS    = ["level1"]
# 환경변수로도 끌 수 있음:  os.environ["WANDB_DISABLED"] = "true"

In [7]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

# Set A 의 train/val 로더 구성
train_ds = BDDAttrDataset(DATA_ROOT, split="train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, split="val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# 속성별 클래스 분포 출력 — 불균형 정도를 직접 확인할 것
for a in ATTRIBUTES:
    print(f"{a:10s} 클래스별 샘플 수 (train) = {train_ds.class_counts(a).tolist()}")

데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=a08ed2f0-1471-4d3c-a491-2769fb3592a0
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:03<00:00, 79.0MB/s] 


압축 해제 중...
완료 → ../data/set_a
weather    클래스별 샘플 수 (train) = [3100, 800, 400, 200, 0, 500]
scene      클래스별 샘플 수 (train) = [3052, 1386, 562]
timeofday  클래스별 샘플 수 (train) = [2479, 2129, 392]


In [7]:
from torch import nn

def train_one(model_fn, name, epochs=20, loss_weights=None):
    """단일 모델을 학습하고 체크포인트와 wandb 산출물을 저장한다."""
    set_seed(SEED, deterministic=True)
    model = model_fn().to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    cfg = TrainConfig(epochs=epochs, loss_weights=loss_weights or {a: 1.0 for a in ATTRIBUTES})

    # wandb 로거 — WANDB_PROJECT 가 None 이거나 wandb 미설치 시 자동 no-op
    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level1-{name}",
        config={
            "backbone": name, "epochs": epochs, "batch": BATCH,
            "lr": 3e-4, "weight_decay": 5e-4, "seed": SEED,
            "loss_weights": cfg.loss_weights,
        },
        tags=WANDB_TAGS + [name],
    )

    trainer = MultiTaskTrainer(model, optim, sched, losses, device, cfg, logger=logger)
    history = trainer.fit(train_loader, val_loader)

    # 학습 종료 후 — 속성별 정규화 confusion matrix 를 wandb 에 업로드
    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    logger.finish()

    os.makedirs("../checkpoints", exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "history": history},
               f"../checkpoints/level1_{name}.pth")
    return model, history

# TODO: 각 모델을 학습하고 최종 Avg-Macro-F1 및 속성별 MF1 을 기록하세요.
vgg_model, vgg_hist = train_one(VGG16,    "vgg16",    epochs=30)
# r18_model, r18_hist = train_one(resnet18, "resnet18", epochs=30)
# r50_model, r50_hist = train_one(resnet50, "resnet50", epochs=30)

print("Final Avg-MF1:", vgg_hist["val_avg_mf1"][-1])
print("Final per-attribute MF1:", vgg_hist["val_per_mf1"][-1])

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[epoch 01/30] train_loss=2.6505  val_avg_MF1=0.3389  per={'weather': 0.125, 'scene': 0.2531017369727047, 'timeofday': 0.6386183261183261}


[epoch 02/30] train_loss=2.1808  val_avg_MF1=0.3789  per={'weather': 0.15048834950855772, 'scene': 0.3452125062179689, 'timeofday': 0.6410792147328345}


[epoch 03/30] train_loss=2.1359  val_avg_MF1=0.4403  per={'weather': 0.192657316465959, 'scene': 0.34464219503151644, 'timeofday': 0.7834978922367162}


[epoch 04/30] train_loss=2.0593  val_avg_MF1=0.4320  per={'weather': 0.2197192479766752, 'scene': 0.31766407889642806, 'timeofday': 0.7585382195231135}


...

[epoch 28/30] train_loss=1.3509  val_avg_MF1=0.5545  per={'weather': 0.4014800122914863, 'scene': 0.4708967223252938, 'timeofday': 0.7912334509809966}


[epoch 29/30] train_loss=1.3614  val_avg_MF1=0.5590  per={'weather': 0.4182031342199401, 'scene': 0.4676011665348519, 'timeofday': 0.7912334509809966}


[epoch 30/30] train_loss=1.3598  val_avg_MF1=0.5595  per={'weather': 0.41683237480651275, 'scene': 0.47031847869112026, 'timeofday': 0.7912334509809966}


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁
val/avg_macro_f1,▁▂▄▄▄▅▆▅▅▅▆▆▆▇▇▇▇▇▇██▇▇███████
val/mf1_scene,▁▄▄▃▂▄▆▅▅▆▆▇▆▇▆▇▇▆▇▇▇▇▇▇▇█████
val/mf1_timeofday,▁▁▇▆▆▇██▇▆██▆▇▇▆▇▇▇▇▇▆▇▇▇▇▇▇▇▇
val/mf1_weather,▁▂▃▃▃▃▃▃▃▄▄▅▆▇▇▇▆▆▇█▇▇▇▇██████
epoch,30
lr,0
train/loss,1.35981
val/avg_macro_f1,0.55946


Final Avg-MF1: 0.5594614348262099
Final per-attribute MF1: {'weather': 0.41683237480651275, 'scene': 0.47031847869112026, 'timeofday': 0.7912334509809966}


In [9]:
from torch import nn

def train_one(model_fn, name, epochs=20, loss_weights=None):
    """단일 모델을 학습하고 체크포인트와 wandb 산출물을 저장한다."""
    set_seed(SEED, deterministic=True)
    model = model_fn().to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    cfg = TrainConfig(epochs=epochs, loss_weights=loss_weights or {a: 1.0 for a in ATTRIBUTES})

    # wandb 로거 — WANDB_PROJECT 가 None 이거나 wandb 미설치 시 자동 no-op
    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level1-{name}",
        config={
            "backbone": name, "epochs": epochs, "batch": BATCH,
            "lr": 3e-4, "weight_decay": 5e-4, "seed": SEED,
            "loss_weights": cfg.loss_weights,
        },
        tags=WANDB_TAGS + [name],
    )

    trainer = MultiTaskTrainer(model, optim, sched, losses, device, cfg, logger=logger)
    history = trainer.fit(train_loader, val_loader)

    # 학습 종료 후 — 속성별 정규화 confusion matrix 를 wandb 에 업로드
    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    logger.finish()

    os.makedirs("../checkpoints", exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "history": history},
               f"../checkpoints/level1_{name}.pth")
    return model, history

# TODO: 각 모델을 학습하고 최종 Avg-Macro-F1 및 속성별 MF1 을 기록하세요.
# vgg_model, vgg_hist = train_one(VGG16,    "vgg16",    epochs=30)
# r18_model, r18_hist = train_one(resnet18, "resnet18", epochs=30)
r50_model, r50_hist = train_one(resnet50, "resnet50", epochs=30)

print("Final Avg-MF1:", r50_hist["val_avg_mf1"][-1])
print("Final per-attribute MF1:", r50_hist["val_per_mf1"][-1])

[epoch 01/30] train_loss=2.4501  val_avg_MF1=0.4183  per={'weather': 0.21885387266060766, 'scene': 0.344293283336258, 'timeofday': 0.6917687590219312}


[epoch 02/30] train_loss=2.0743  val_avg_MF1=0.4435  per={'weather': 0.21423519976669583, 'scene': 0.3662230877420751, 'timeofday': 0.7501422471452086}


[epoch 03/30] train_loss=1.9808  val_avg_MF1=0.3520  per={'weather': 0.2702560802560802, 'scene': 0.3015386776448723, 'timeofday': 0.48428030303030306}


[epoch 04/30] train_loss=1.9133  val_avg_MF1=0.4905  per={'weather': 0.271061438977481, 'scene': 0.40279992911571866, 'timeofday': 0.7975016603554531}


[epoch 05/30] train_loss=1.8417  val_avg_MF1=0.4721  per={'weather': 0.24437927914968882, 'scene': 0.41042129349215967, 'timeofday': 0.761363008413901}


...

[epoch 28/30] train_loss=1.0846  val_avg_MF1=0.6323  per={'weather': 0.5146662976572808, 'scene': 0.5742308537816985, 'timeofday': 0.8080925763800031}


[epoch 29/30] train_loss=1.0457  val_avg_MF1=0.6350  per={'weather': 0.5134144922001872, 'scene': 0.5833659927671303, 'timeofday': 0.8080925763800031}


[epoch 30/30] train_loss=1.0684  val_avg_MF1=0.6332  per={'weather': 0.5186396709902136, 'scene': 0.5825455271283214, 'timeofday': 0.798346366894754}


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
val/avg_macro_f1,▃▃▁▄▄▅▅▄▃▅▅▆▅▆▆▆▆▆▆▆▆▇▇█▇▇████
val/mf1_scene,▂▃▁▄▄▄▃▄▁▄▄▅▃▅▅▄▃▅▃▄▅▅▆▇▆▇▇███
val/mf1_timeofday,▅▆▁▇▇▇▇▆▆▇▇▇▇█▇▇█▇█▇▇▇▇▇█▇▇██▇
val/mf1_weather,▁▁▂▂▂▃▅▂▃▅▄▅▆▅▅▆▆▇▇▇▇▇▆█▆▇████
epoch,30
lr,0
train/loss,1.06836
val/avg_macro_f1,0.63318


Final Avg-MF1: 0.633177188337763
Final per-attribute MF1: {'weather': 0.5186396709902136, 'scene': 0.5825455271283214, 'timeofday': 0.798346366894754}


## 분석 (리포트 필수 포함 항목)

1. **수렴 비교**: VGG-16 과 ResNet-18 의 학습 손실 곡선을 함께 그리세요. Skip connection 이 없는 깊은 네트워크가 정체되는 현상이 관찰되는지, 그 원인은 무엇인지 서술하세요.
2. **속성별 MF1 표**: 각 백본에 대해 Weather / Scene / Time of Day 의 Macro-F1 을 표로 정리하세요. 어느 속성이 가장 어려운지, 깊이 변화에 따라 그 양상이 어떻게 바뀌는지 분석하세요.
3. **Loss 가중치 민감도**: ResNet-18 에 대해 최소 두 가지 비자명한 가중치 설정 (예: `(1, 1, 1)` vs `(2, 1, 1)`) 으로 재학습하고 Avg-MF1 변화를 보고하세요.

wandb 를 활성화한 경우 같은 프로젝트의 Run 들을 비교하면 학습 곡선·속성별 MF1·confusion matrix 가 한눈에 정리됩니다.

## 리포트용 표/그림 PNG 저장

아래 셀은 앞에서 학습한 `*_hist` 변수 또는 `../checkpoints/level1_*.pth` 파일을 읽어서 분석에 필요한 표와 그림을 `../reports/level1/`에 저장합니다.

In [ ]:
# 리포트용 표/그림 PNG 자동 저장
# 필요 시 분석 3 실험 예시:
# r18_model, r18_hist = train_one(resnet18, 'resnet18', epochs=30)
# r18_w211_model, r18_w211_hist = train_one(
#     resnet18, 'resnet18_w211', epochs=30,
#     loss_weights={'weather': 2.0, 'scene': 1.0, 'timeofday': 1.0},
# )

from pathlib import Path
import os
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

REPORT_DIR = Path('../reports/level1')
CHECKPOINT_DIR = Path('../checkpoints')
REPORT_DIR.mkdir(parents=True, exist_ok=True)

ATTR_DISPLAY = {
    'weather': 'Weather MF1',
    'scene': 'Scene MF1',
    'timeofday': 'Time of Day MF1',
}

saved_files = []
missing_items = []

def load_history(history_vars, checkpoint_names):
    for var_name in history_vars:
        hist = globals().get(var_name)
        if hist is not None:
            return hist, f'memory:{var_name}'
    for ckpt_name in checkpoint_names:
        ckpt_path = CHECKPOINT_DIR / ckpt_name
        if ckpt_path.exists():
            try:
                obj = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            except TypeError:
                obj = torch.load(ckpt_path, map_location='cpu')
            return obj.get('history', obj), str(ckpt_path)
    return None, None

def final_or_nan(values):
    return float(values[-1]) if values else np.nan

def best_epoch_and_score(values):
    if not values:
        return np.nan, np.nan
    best_idx = int(np.argmax(values))
    return best_idx + 1, float(values[best_idx])

def history_to_row(label, hist, source):
    per_history = hist.get('val_per_mf1', [])
    per = per_history[-1] if per_history else {}
    best_epoch, best_avg = best_epoch_and_score(hist.get('val_avg_mf1', []))
    row = {
        'Model': label,
        'Epochs': len(hist.get('train_loss', [])),
        'Final Train Loss': final_or_nan(hist.get('train_loss', [])),
        'Final Avg-MF1': final_or_nan(hist.get('val_avg_mf1', [])),
        'Best Avg-MF1': best_avg,
        'Best Epoch': best_epoch,
    }
    for attr in ATTRIBUTES:
        row[ATTR_DISPLAY.get(attr, attr)] = float(per.get(attr, np.nan))
    row['Source'] = source or ''
    return row

def format_for_display(df):
    out = df.copy()
    integer_cols = {'Epochs', 'Best Epoch', 'Class Index', 'Class Count'}
    for col in out.columns:
        if col in integer_cols:
            out[col] = out[col].map(lambda x: '' if pd.isna(x) else f'{int(x)}')
        elif pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].map(lambda x: '' if pd.isna(x) else f'{float(x):.4f}')
    return out

def save_df_png(df, png_path, title):
    display_df = format_for_display(df)
    fig_width = max(8, min(18, 1.35 * len(display_df.columns)))
    fig_height = max(2.4, 0.55 * (len(display_df) + 1) + 1.0)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis('off')
    ax.set_title(title, fontsize=14, weight='bold', pad=14)
    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        loc='center',
        cellLoc='center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.35)
    table.auto_set_column_width(col=list(range(len(display_df.columns))))
    for (row, _), cell in table.get_celld().items():
        cell.set_edgecolor('#d0d7de')
        if row == 0:
            cell.set_facecolor('#24292f')
            cell.set_text_props(color='white', weight='bold')
        else:
            cell.set_facecolor('#f6f8fa' if row % 2 == 0 else 'white')
    fig.tight_layout()
    fig.savefig(png_path, dpi=220, bbox_inches='tight')
    plt.close(fig)

def save_table_outputs(df, stem, title):
    if df.empty:
        return
    csv_path = REPORT_DIR / f'{stem}.csv'
    png_path = REPORT_DIR / f'{stem}.png'
    df.to_csv(csv_path, index=False)
    save_df_png(df, png_path, title)
    saved_files.extend([csv_path, png_path])

model_specs = [
    ('VGG-16', ['vgg_hist'], ['level1_vgg16.pth']),
    ('ResNet-18', ['r18_hist'], ['level1_resnet18.pth']),
    ('ResNet-50', ['r50_hist'], ['level1_resnet50.pth']),
]

histories = {}
model_rows = []
for label, hist_vars, ckpts in model_specs:
    hist, source = load_history(hist_vars, ckpts)
    if hist is None:
        missing_items.append(f'{label}: {hist_vars} or {ckpts}')
        continue
    histories[label] = hist
    model_rows.append(history_to_row(label, hist, source))

model_df = pd.DataFrame(model_rows)
save_table_outputs(model_df, 'level1_model_mf1_summary', 'Model Avg-MF1 and Per-Attribute MF1')
if not model_df.empty:
    display(model_df)

convergence_rows = []
for label in ['VGG-16', 'ResNet-18']:
    hist = histories.get(label)
    if hist is None:
        continue
    losses = hist.get('train_loss', [])
    start_loss = float(losses[0]) if losses else np.nan
    final_loss = final_or_nan(losses)
    loss_drop = start_loss - final_loss if np.isfinite(start_loss) and np.isfinite(final_loss) else np.nan
    loss_drop_pct = (loss_drop / start_loss) * 100 if np.isfinite(loss_drop) and start_loss != 0 else np.nan
    best_epoch, best_avg = best_epoch_and_score(hist.get('val_avg_mf1', []))
    convergence_rows.append({
        'Model': label,
        'Epochs': len(losses),
        'Start Train Loss': start_loss,
        'Final Train Loss': final_loss,
        'Min Train Loss': float(np.min(losses)) if losses else np.nan,
        'Loss Drop': loss_drop,
        'Loss Drop %': loss_drop_pct,
        'Best Avg-MF1': best_avg,
        'Best Epoch': best_epoch,
    })

convergence_df = pd.DataFrame(convergence_rows)
save_table_outputs(convergence_df, 'level1_convergence_summary', 'VGG-16 vs ResNet-18 Convergence Summary')
if not convergence_df.empty:
    display(convergence_df)

if all(label in histories for label in ['VGG-16', 'ResNet-18']):
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for label in ['VGG-16', 'ResNet-18']:
        losses = histories[label].get('train_loss', [])
        ax.plot(range(1, len(losses) + 1), losses, marker='o', linewidth=2, label=label)
    ax.set_title('Training Loss Convergence: VGG-16 vs ResNet-18')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Train Loss')
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    loss_curve_path = REPORT_DIR / 'level1_vgg_vs_resnet18_train_loss.png'
    fig.savefig(loss_curve_path, dpi=220, bbox_inches='tight')
    plt.close(fig)
    saved_files.append(loss_curve_path)

sensitivity_specs = [
    ('ResNet-18', '(1, 1, 1)', ['r18_hist', 'r18_w111_hist'], ['level1_resnet18.pth', 'level1_resnet18_w111.pth']),
    ('ResNet-18', '(2, 1, 1)', ['r18_w211_hist'], ['level1_resnet18_w211.pth']),
]

sensitivity_rows = []
for label, weights, hist_vars, ckpts in sensitivity_specs:
    hist, source = load_history(hist_vars, ckpts)
    if hist is None:
        missing_items.append(f'{label} loss weights {weights}: {hist_vars} or {ckpts}')
        continue
    row = history_to_row(label, hist, source)
    row['Loss Weights'] = weights
    sensitivity_rows.append(row)

sensitivity_df = pd.DataFrame(sensitivity_rows)
if not sensitivity_df.empty:
    baseline = sensitivity_df['Final Avg-MF1'].iloc[0]
    sensitivity_df['Delta Final Avg-MF1 vs baseline'] = sensitivity_df['Final Avg-MF1'] - baseline
    ordered_cols = [
        'Model', 'Loss Weights', 'Final Avg-MF1', 'Delta Final Avg-MF1 vs baseline',
        'Best Avg-MF1', 'Best Epoch', 'Weather MF1', 'Scene MF1', 'Time of Day MF1', 'Source'
    ]
    sensitivity_df = sensitivity_df[[c for c in ordered_cols if c in sensitivity_df.columns]]
save_table_outputs(sensitivity_df, 'level1_loss_weight_sensitivity', 'ResNet-18 Loss Weight Sensitivity')
if not sensitivity_df.empty:
    display(sensitivity_df)

if 'train_ds' in globals():
    class_rows = []
    for attr in ATTRIBUTES:
        counts = train_ds.class_counts(attr).tolist()
        names = CLASS_NAMES.get(attr, [str(i) for i in range(len(counts))])
        for idx, count in enumerate(counts):
            class_rows.append({
                'Attribute': attr,
                'Class Index': idx,
                'Class Name': names[idx] if idx < len(names) else str(idx),
                'Class Count': int(count),
            })
    class_df = pd.DataFrame(class_rows)
    save_table_outputs(class_df, 'level1_train_class_distribution', 'Set A Train Class Distribution')
    display(class_df)
else:
    missing_items.append('train_ds: class distribution table')

config_df = pd.DataFrame([
    {'Item': 'Seed', 'Value': globals().get('SEED', 'not run')},
    {'Item': 'Batch Size', 'Value': globals().get('BATCH', 'not run')},
    {'Item': 'Device', 'Value': str(globals().get('device', 'not run'))},
    {'Item': 'Data Root', 'Value': globals().get('DATA_ROOT', 'not run')},
    {'Item': 'Checkpoint Dir', 'Value': str(CHECKPOINT_DIR)},
    {'Item': 'Report Dir', 'Value': str(REPORT_DIR)},
])
save_table_outputs(config_df, 'level1_experiment_config', 'Level 1 Experiment Config')
display(config_df)

required_outputs_df = pd.DataFrame([
    {'Analysis': '1. Convergence', 'Saved Output': 'level1_convergence_summary.png / level1_vgg_vs_resnet18_train_loss.png', 'Needs': 'VGG-16, ResNet-18 histories'},
    {'Analysis': '2. Per-attribute MF1', 'Saved Output': 'level1_model_mf1_summary.png', 'Needs': 'VGG-16, ResNet-18, ResNet-50 histories'},
    {'Analysis': '3. Loss weights', 'Saved Output': 'level1_loss_weight_sensitivity.png', 'Needs': 'ResNet-18 baseline and weighted histories'},
    {'Analysis': 'Dataset check', 'Saved Output': 'level1_train_class_distribution.png', 'Needs': 'train_ds loaded'},
])
save_table_outputs(required_outputs_df, 'level1_required_output_index', 'Level 1 Required Output Index')
display(required_outputs_df)

if missing_items:
    missing_df = pd.DataFrame({'Missing Item': missing_items})
    save_table_outputs(missing_df, 'level1_missing_results', 'Missing Results To Generate')
    print('아직 생성되지 않은 결과가 있습니다. 필요한 학습 셀을 실행한 뒤 이 셀을 다시 실행하세요.')
    display(missing_df)

print('Saved report files:')
for path in saved_files:
    print('-', path)
